<a href="https://colab.research.google.com/github/emmakowu3579-ui/ATF-Challenge-Notebooks/blob/main/04_prompting_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1.4: Prompt Engineering for Digital Twins

## Learning Objectives
- Master different prompting techniques
- Understand zero-shot, few-shot, and chain-of-thought
- Apply prompting to digital twin interactions

## Key Insight
> 'Moving from Chat to System Instructions'

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict

print('✅ Ready for prompting exercises!')

✅ Ready for prompting exercises!


## Part 1: Prompting Fundamentals

### 1.1 Zero-Shot Prompting

In [2]:
# Example prompts for digital twin
zero_shot_examples = [
    {
        'task': 'activity_prediction',
        'prompt': 'Predict the next activity: I just finished work, had dinner, and now...',
        'expected': 'leisure/relaxation activity'
    },
    {
        'task': 'preference_extraction',
        'prompt': 'Extract preferences from: I love outdoor activities but hate crowded places',
        'expected': {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}
    }
]

print('Zero-Shot Examples:')
for ex in zero_shot_examples:
    print(f"\nTask: {ex['task']}")
    print(f"Prompt: {ex['prompt']}")
    print(f"Expected: {ex['expected']}")

Zero-Shot Examples:

Task: activity_prediction
Prompt: Predict the next activity: I just finished work, had dinner, and now...
Expected: leisure/relaxation activity

Task: preference_extraction
Prompt: Extract preferences from: I love outdoor activities but hate crowded places
Expected: {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}


### 🎯 Exercise 1.1 (Easy): Create Your Prompts

**Task**: Design prompts for your digital twin use case.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict

print('✅ Ready for prompting exercises!')


# Set up Groq
!pip install -q groq

import os
from groq import Groq
import json
from google.colab import userdata


MODEL_NAME = "openai/gpt-oss-120b"

# Read API key from Colab Secrets
# In Colab: click the key icon (Secrets) and create GROQ_API_KEY
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

print("Connected. Ready.")

client = Groq(
    api_key=API_KEY,
)

def llm(prompt: str) -> str:
  """Simple helper for single-turn prompts."""
  chat_completion = client.chat.completions.create(
      model=MODEL_NAME,
      messages=[
          {"role": "user", "content": prompt}
      ]
  )
  return chat_completion.choices[0].message.content # Changed .messages.content to .message.content

print("Connected. Ready.")

✅ Ready for prompting exercises!
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.8 MB/s eta 0:00:00
Connected. Ready.
Connected. Ready.


In [4]:
prompt = '''
Predict the next activity.
I just woke up from bed, had breakfast,
and now
'''

response = llm(prompt)
print(response)

You’re probably about to start getting ready for the day—maybe brushing your teeth, taking a quick shower, or checking your phone before heading out to work/school or whatever you have planned.


## Few-Shot: Show examples, do not explain

In [5]:
prompt = '''
Examples:
Input: finished gym, lunch, coffee
Output: {"next":"Work","confidence":0.82}

Input: Dinner, watched TV
Output: {"next":"Sleep","confidence":0.91}

Input: Just woke up, washed down
Output: {"next":"Bible study", "confidence": 0.90}

Input: washed down, eat breakfast, now
Output: '''

response = llm(prompt)
print(response)

{"next":"Work","confidence":0.84}


## CoT vs Plain
Chain-of-thought: four extra words

In [6]:
plain = '''Should I exercise today?
Context: worked 10h, slept 5h.'''

cot = '''Should I exercise today?
Context: worked 10h, slept 5h.
Think step by step.'''

print('--- Plain ---')
print(llm(plain))
print()
print('--- Chain-of-Thought ---')
print(llm(cot))

--- Plain ---
**Short answer:** Yes—if you feel up to it, a light‑to‑moderate workout can actually boost your energy and mood after a long day and a short night of sleep. But you don’t need to push yourself into a hard‑core session; listening to your body is key.

---

## Why a gentle workout can help

| Benefit | How it applies after a 10‑hour workday & 5‑hour sleep |
|--------|-------------------------------------------------------|
| **Increases alertness** | Light cardio (a brisk walk, easy bike ride) raises heart rate and releases norepinephrine, which helps shake off that “groggy” feeling. |
| **Improves mood** | Exercise triggers endorphins and serotonin, counteracting the stress hormones that build up during long work hours. |
| **Supports recovery** | Gentle movement promotes blood flow to muscles, helping to flush out metabolic waste that can build up when you sit for many hours. |
| **Boosts sleep quality** | Even modest activity can deepen the next night’s sleep, making up 

In [9]:
# Build the Twin's System Instruction

system = '''
ROLE: You are a digital twin for (user):
You represent their preferences and behavioural patterns.

KNOWLEDGE: You have the user's activity history and temporal patterns.

CONSTRAINTS: Always return structured output. Never guess. If data is missing,
say so explicitly.

TONE: Precise, analytical.
'''
def llm(prompt: str) -> str:
  chat_completion = client.chat.completions.create(
      model=MODEL_NAME,
      messages=[
          {'role': 'system', 'content':system},
          {'role':'user', 'content': prompt}
      ]
  )
  return chat_completion.choices[0].message.content

print(llm('What next?'))

{
  "status": "error",
  "message": "Insufficient context to determine the appropriate next step.",
  "required_information": [
    "description of the current task or project",
    "specific goal or outcome you are aiming for",
    "any recent actions or decisions that precede this point"
  ],
  "suggested_action": "Please provide the missing details so I can generate a precise recommendation."
}


In [14]:
# Twin system instructions

user = "Kofi"

user_profile = {
    "name": "Kofi",
    "role": "Post-grad student",
    "preferences": [
        "prefers analytical explanations",
        "likes structured responses",
        "works often with Python and presentations"
    ],
    "goals": [
        "improve programming skills",
        "build more AI models for Clinical integration",
        "build Data analysis skills and engage in research"
    ]
}

activity_history = [
    {"time": "2026-07-10", "activity": "Did a course under Data Science in DataCamp"},
    {"time": "2026-07-12", "activity": "Train AI model for a project in AI-based TB detection"},
    {"time": "2026-07-14", "activity": "Joined classmates to do a class assignment"}
]

temporal_patterns = {
    "most_active_period": "evenings",
    "common_tasks": [
        "learning about Data Science",
        "engaged in class activities and assignments",
        "engaged in AI projects and research publications"
    ],
    "recurring_focus": [
        "emerging technologies integrated into clinical workflows",
        "AI-based softwares for diagnosis and monitoring of health conditions"
    ]
}

system = f'''
ROLE: You are a digital twin for {user}.
You simulate the user's likely priorities, preferences, and next actions
using only the evidence provided.

AVAILABLE DATA:
1. User profile: {user_profile}
2. Activity history: {activity_history}
3. Temporal patterns: {temporal_patterns}

OBJECTIVE:
Given a user query, produce an informed and structured assessment of:
- the user's likely current context,
- the most probable next actions,
- the evidence supporting each action,
- any missing data that limits confidence.

REASONING RULES:
- Use only the supplied data.
- Do not invent facts.
- Separate observed facts from inferred conclusions.
- If evidence is weak or missing, say so explicitly.
- Rank likely next actions by confidence.
- Ground every inference in profile, history, or temporal patterns.

OUTPUT FORMAT:
Return valid JSON with this structure:
{{
  "user": "string",
  "current_context": {{
    "summary": "string",
    "observed_signals": ["string"]
    }},
  "predicted_next_actions": [
    {{
      "action": "string",
      "confidence": "high | medium | low",
      "supporting_data": ["string"]
    }}
  ],
  "missing_data": ["string"],
  "response_quality": {{
    "grounded_in_data": true,
    "contains_guessing": false
  }}
}}

TONE: Precise, analytical, concise.
'''

r = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": "What next?"}
    ],
    response_format={"type": "json_object"}
)

print(r.choices[0].message.content)

{
  "user": "Kofi",
  "current_context": {
    "summary": "Kofi has recently completed a Data Science course, trained an AI model for TB detection, and participated in a class assignment. Evenings are his most active period, focusing on learning, AI projects, and class work related to clinical AI integration.",
    "observed_signals": [
      "Completed DataCamp Data Science course on 2026-07-10",
      "Trained AI model for TB detection on 2026-07-12",
      "Joined classmates for a class assignment on 2026-07-14",
      "Prefers analytical, structured work and uses Python frequently",
      "Goals include improving programming, building AI models for clinical use, and enhancing data analysis skills"
    ]
  },
  "predicted_next_actions": [
    {
      "action": "Refine and evaluate the AI TB detection model (e.g., hyperparameter tuning, validation on new data)",
      "confidence": "high",
      "supporting_data": [
        "Most recent activity was training an AI model for TB detect

In [ ]:
# YOUR CODE HERE
my_prompts = [
    # Add your prompts
]